# Statistical Analysis of Yine Prosodic Prominence (Bayesian & ML)

**Objective:** To statistically determine the acoustic correlates of primary and secondary stress in Yine and to test specific linguistic hypotheses regarding cue trading, positional effects, and intonational interaction.

**Methodology:**
This script implements a dual-pronged analytical approach:
1.  **Confirmatory Analysis (Bayesian Multilevel Modeling):** Uses `brms` to fit a series of increasingly complex Cumulative Link Mixed Models (CLMMs) to test specific hypotheses. Models are compared using Leave-One-Out Information Criterion (LOOIC).
2.  **Exploratory Analysis (Random Forests):** Uses Conditional Inference Forests (`party::cforest`) to provide an unbiased ranking of cue importance (Cue Weighting), robust to multicollinearity.

**Environment:**
*   **System:** Local / WSL (Linux)
*   **Languages:** Python (Data Management) + R (Statistical Modeling via `rpy2`)
*   **Input:** Preprocessed CSV datasets.
*   **Output:** Tables (CSV), Model Summaries (Text), and Plots (PNG).

## Part 1: Environment Setup and Data Loading
This section initializes the Python and R environments, defines local file paths, and loads the preprocessed metric data from CSV files. It ensures that all necessary libraries (`brms`, `party`, `ggplot2`, etc.) are loaded in the R instance.

In [ ]:
# ==============================================================================
# 1. SETUP: INSTALL PYTHON LIBRARIES
# ==============================================================================
# In your WSL terminal, ensure you have run:
# pip install pandas rpy2 scikit-learn statsmodels

import sys
print(f"Python Version: {sys.version}")

In [ ]:
# ==============================================================================
# 2. CONFIGURATION (LOCAL WSL VERSION)
# ==============================================================================
import os

# --- Local File Paths ---
# Since this notebook is inside 'notebooks/', we use '..' to go up to the project root.
PROJECT_ROOT = '..' 
DATA_DIR = os.path.join(PROJECT_ROOT, 'data')

# Input: Where the CSV from Script 5 lives
INPUT_DIR = os.path.join(DATA_DIR, 'processed')

# Output: Where statistical results/plots/summaries will go
OUTPUT_DIR = os.path.join(DATA_DIR, 'analysis_results')
MODEL_OUTPUT_FOLDER = os.path.join(OUTPUT_DIR, 'Model_Summaries')

# --- Input Files (CSVs) ---
# This matches the filename created by Script 5
INPUT_FILES = ['yine_stats_metrics.csv']

# --- Output Files (CSVs) ---
VIF_OUTPUT_FILE = os.path.join(OUTPUT_DIR, 'Collinearity_Check_VIF.csv')
BRM_OUTPUT_FILE = os.path.join(OUTPUT_DIR, 'BRM_Coefficients.csv')
RF_FULL_OUTPUT_FILE = os.path.join(OUTPUT_DIR, 'RF_Results_Full_Model.csv')
RF_ACOUSTIC_OUTPUT_FILE = os.path.join(OUTPUT_DIR, 'RF_Results_Acoustic_Only.csv')

# Create output directories if they don't exist
# We don't create INPUT_DIR because it must already exist for the script to run
os.makedirs(OUTPUT_DIR, exist_ok=True)
os.makedirs(MODEL_OUTPUT_FOLDER, exist_ok=True)

print("Configuration set:")
print(f"  Input Directory:  '{os.path.abspath(INPUT_DIR)}'")
print(f"  Output Directory: '{os.path.abspath(OUTPUT_DIR)}'")
print(f"  Model Summaries:  '{os.path.abspath(MODEL_OUTPUT_FOLDER)}'")

In [ ]:
# ==============================================================================
# 3. DATA LOADING AND CONCATENATION (LOCAL CSV + SANITIZATION)
# ==============================================================================
import pandas as pd
import glob
import numpy as np

def load_and_concatenate_data(input_folder, file_list):
    """
    Loads data from local CSV files, concatenates them, and performs
    type sanitization for R compatibility.
    """
    print(f"\n--- Loading Data from '{input_folder}' ---")
    all_dfs = []
    
    for filename in file_list:
        file_path = os.path.join(input_folder, filename)
        if os.path.exists(file_path):
            print(f"  - Reading '{filename}'...")
            try:
                df = pd.read_csv(file_path)
                all_dfs.append(df)
                print(f"    -> Loaded {len(df)} rows.")
            except Exception as e:
                print(f"    !!! Error reading file: {e}")
        else:
            print(f"    !!! File not found: {file_path}")

    if not all_dfs:
        print("!!! CRITICAL ERROR: No data loaded. Halting.")
        return None

    main_df = pd.concat(all_dfs, ignore_index=True)
    print(f"\n--- Concatenation complete. Total rows: {len(main_df)} ---")

    # --- Final Data Type Conversion (Standardization) ---
    z_score_cols = [col for col in main_df.columns if col.endswith('_z')]
    print(f"Converting {len(z_score_cols)} standardized predictor columns to numeric...")
    for col in z_score_cols:
        main_df[col] = pd.to_numeric(main_df[col], errors='coerce')

    main_df.dropna(subset=z_score_cols, inplace=True)
    
    # --- R COMPATIBILITY SANITIZATION ---
    print("Sanitizing data types for R compatibility...")
    
    # 1. Convert BOOL to INT (0/1)
    bool_cols = main_df.select_dtypes(include=['bool']).columns
    for col in bool_cols:
        main_df[col] = main_df[col].astype(int)
    
    # 2. Convert INT64 to FLOAT (R integers are 32-bit; int64 can overflow)
    int_cols = main_df.select_dtypes(include=['int64']).columns
    for col in int_cols:
        main_df[col] = main_df[col].astype(float)
        
    # 3. Ensure Object columns are clean Strings
    obj_cols = main_df.select_dtypes(include=['object']).columns
    for col in obj_cols:
        main_df[col] = main_df[col].astype(str)
        
    print(f"Data preparation complete. Final rows: {len(main_df)}")
    return main_df

# --- Execute ---
main_df = load_and_concatenate_data(INPUT_DIR, INPUT_FILES)

In [ ]:
# ==============================================================================
# 4. DATA TYPE VERIFICATION & STRESS DISTRIBUTION CHECK
# ==============================================================================

if 'main_df' in locals() and main_df is not None:
    # 1. Print Data Types and Memory Usage
    print("--- DataFrame Info ---")
    print(main_df.info())

    # 2. Print Stress Category Distribution
    if 'Matteson_Stress' in main_df.columns:
        print("\n--- Distribution of Matteson_Stress Categories ---")

        # Calculate counts
        counts = main_df['Matteson_Stress'].value_counts()

        # Calculate percentages
        percentages = main_df['Matteson_Stress'].value_counts(normalize=True) * 100

        # Combine into a readable DataFrame
        dist_df = pd.DataFrame({
            'Count': counts,
            'Percentage (%)': percentages.round(2)
        })

        print(dist_df)

        # Print the specific ratios relevant to your upcoming binary models
        try:
            n_primary = counts.get('Primary_stress', 0)
            n_secondary = counts.get('Secondary_stress', 0)
            n_unstressed = counts.get('Unstressed', 0)

            print("\n--- Imbalance Ratios for Binary Models ---")
            if n_secondary > 0:
                ratio_p_s = n_primary / n_secondary
                print(f"1. Primary vs. Secondary:  {n_primary} vs {n_secondary} (Ratio: {ratio_p_s:.2f} to 1)")

                ratio_u_s = n_unstressed / n_secondary
                print(f"2. Unstressed vs. Secondary: {n_unstressed} vs {n_secondary} (Ratio: {ratio_u_s:.2f} to 1)")
            else:
                print("Warning: No Secondary_stress found.")

        except Exception as e:
            print(f"Could not calculate ratios: {e}")

else:
    print("!!! ERROR: 'main_df' not found. Please run the data loading cell first. !!!")

In [ ]:
# ==============================================================================
# 5. DATA PREPARATION FOR MODELING
# ==============================================================================
def prepare_data_for_modeling(df):
    print("\n--- Preparing DataFrame for Modeling ---")
    print("  - No global data preparations needed at this step.")
    return df

if 'main_df' in locals() and main_df is not None:
    main_df = prepare_data_for_modeling(main_df)

In [ ]:
# ==============================================================================
# 6. R Environment Setup (Simplified)
# ==============================================================================
%load_ext rpy2.ipython

import rpy2.robjects as ro
from rpy2.robjects import pandas2ri
from rpy2.robjects.conversion import localconverter

# Activate pandas conversion
pandas2ri.activate()

def setup_r_environment(df):
    """
    Transfers the pre-sanitized DataFrame to the R environment.
    """
    print("\n--- Transferring Data to R Environment ---")
    
    try:
        # We use a context manager to enforce the pandas conversion rules
        with localconverter(ro.default_converter + pandas2ri.converter):
            # Explicitly convert Python DataFrame -> R DataFrame
            r_dataframe = ro.conversion.py2rpy(df)
            
            # Assign the converted object to the R global environment
            ro.globalenv['r_df'] = r_dataframe
            
        print("  - Success: DataFrame available in R as 'r_df'.")
        
    except Exception as e:
        print(f"!!! Error transferring data to R: {e} !!!")

# Execute
if 'main_df' in locals() and main_df is not None:
    setup_r_environment(main_df)
else:
    print("!!! Error: main_df not found. Run data loading first.")

In [ ]:
# ==============================================================================
# 7. CHECK R PACKAGES (ROBUST LOCAL VERSION - FIXED LOGIC)
# ==============================================================================
import rpy2.robjects as ro
from rpy2.robjects import pandas2ri
from rpy2.robjects.conversion import localconverter

print("Checking R packages...")

r_code = """
    tryCatch({
        # Suppress startup messages to keep output clean
        suppressPackageStartupMessages({
            library(lme4)
            library(lmerTest)
            library(car)
            library(ordinal)
            library(brms)
            library(loo)
            library(ggplot2)
            library(party)
            library(partykit)
            library(permimp)
            library(caret)
            library(e1071)
            library(pROC)
        })
        cat("All packages loaded successfully.\\n")
        TRUE 
    }, error = function(e) {
        cat(paste("R Error:", e$message, "\\n"))
        FALSE 
    })
"""

try:
    with localconverter(ro.default_converter + pandas2ri.converter):
        # Execute R code
        result_vector = ro.r(r_code)
        
        # Check the actual boolean value inside the R vector
        # result_vector[0] will be 1 (True) or 0 (False)
        if result_vector[0]:
            print("✅ R packages verification passed.")
        else:
            print("❌ R packages verification failed. Check the R output above.")
        
except Exception as e:
    print(f"!!! Python/R Bridge Error: {e} !!!")

## Part 2: Pre-Modeling Diagnostics (Collinearity)
Before fitting complex models, we assess multicollinearity among the predictors using the **Variance Inflation Factor (VIF)**.
*   **Tool:** `car::vif` (Generalized VIF for categorical variables).
*   **Goal:** To ensure that no predictors are so highly correlated that they destabilize the regression estimates.

In [ ]:
# ==============================================================================
# 8. PRE-MODELING CHECK: COLLINEARITY (GVIF VIA R) - FIXED
# ==============================================================================
import rpy2.robjects as ro
from rpy2.robjects import pandas2ri
from rpy2.robjects.conversion import localconverter
pandas2ri.activate()

def check_collinearity_R(df, continuous_predictors, categorical_predictors):
    print("\n--- Running Collinearity Check (VIF/GVIF) ---")
    try:
        # 1. Prepare data in Python
        temp_df = df.copy()
        temp_df = pd.get_dummies(temp_df, columns=categorical_predictors, drop_first=True)
        dummy_cols = [col for col in temp_df.columns if any(cat in col for cat in categorical_predictors)]
        all_model_predictors = continuous_predictors + dummy_cols

        formula = f"{continuous_predictors[0]} ~ " + " + ".join(all_model_predictors[1:])

        r_script = f"""
            library(car)
            temp_model <- lm({formula}, data=r_vif_df)
            vif_results <- vif(temp_model)
            vif_df <- data.frame(Predictor = names(vif_results), VIF_Score = unname(vif_results))
            vif_df
        """

        # 2. Explicit Conversion and Execution
        with localconverter(ro.default_converter + pandas2ri.converter):
            # Explicitly convert Python DF -> R DF
            r_temp_df = ro.conversion.py2rpy(temp_df)

            # Assign to R environment
            ro.globalenv['r_vif_df'] = r_temp_df

            # Run script
            vif_r_df = ro.r(r_script)

            # Explicitly convert Result R DF -> Python DF
            return ro.conversion.rpy2py(vif_r_df)

    except Exception as e:
        print(f"!!! VIF Error: {e} !!!")
        return None

In [ ]:
# ==============================================================================
# 9. EXECUTE COLLINEARITY CHECK
# ==============================================================================
if __name__ == '__main__':
    if 'main_df' in locals() and main_df is not None:
        continuous_predictors = [
            'Duration_z', 'Pitch_z', 'Intensity_Q2_z', 'Spectral_Tilt_DB_z',
            'F1_z', 'F2_z', 'Pace_SPS_z', 'Syllable_position_z', 'Syllable_Index_z',
        ]
        categorical_predictors = [
            'Phrase_Position', 'Is_Interrogative', 'Is_Exclamative',
            'Preceded_by_Silence', 'Followed_by_Silence', 'Vowel'
        ]

        vif_scores_df = check_collinearity_R(main_df, continuous_predictors, categorical_predictors)

        if vif_scores_df is not None:
            vif_scores_df_sorted = vif_scores_df.sort_values(by='VIF_Score', ascending=False)
            print("\n--- VIF Results ---")
            print(vif_scores_df_sorted.to_string(index=False))

            # Save to local CSV
            vif_scores_df_sorted.to_csv(VIF_OUTPUT_FILE, index=False)
            print(f"\nResults saved to: {VIF_OUTPUT_FILE}")

## Part 3: Confirmatory Analysis (Bayesian Multilevel Models)
This is the core hypothesis-testing phase. We use **`brms`** to fit Ordinal Bayesian models (`family=cumulative`).

**The Iterative Workflow:**
We build the model step-by-step to justify the inclusion of each component:
1.  **Model 0:** Baseline (Fixed Effects only).
2.  **Model 1 (a-d):** Add Random Intercepts (controlling for Speaker, Word, File, Utterance).
3.  **Model 2:** Add Random Slopes (allowing cue effects to vary by Speaker).
4.  **Model 3 (a-e):** Test specific blocks of interactions (Positional, Intonational, Cue Trading).
5.  **Model 4:** The Final Model combining all significant structures.

**Comparison:** Models are compared using **LOOIC** (Leave-One-Out Information Criterion) to ensure that added complexity actually improves predictive accuracy.

In [ ]:
# ==============================================================================
# 10. DATA SAMPLING (OPTIONAL)
# ==============================================================================
# Uncomment to use a sample for testing
# SAMPLE_SIZE = 5000
# if 'main_df' in locals() and len(main_df) > SAMPLE_SIZE:
#     print(f"--- Sampling {SAMPLE_SIZE} rows ---")
#     df_sample = main_df.sample(n=SAMPLE_SIZE, random_state=42)
#     ro.r.assign("r_df", df_sample)
#     print("r_df updated with sample.")

In [ ]:
# ==============================================================================
# 11. HELPER: APPEND TO CSV (Simulates Worksheet Appending)
# ==============================================================================
import csv

def append_results_to_csv(file_path, header_text, df):
    """
    Appends a dataframe to a CSV file with a custom header line and blank rows.
    """
    # 1. Write the Section Header and Blank Rows
    with open(file_path, 'a', newline='') as f:
        writer = csv.writer(f)
        writer.writerow([]) # Blank row
        writer.writerow([]) # Blank row
        writer.writerow([header_text]) # Section Title

    # 2. Append the Dataframe (with headers)
    df.to_csv(file_path, mode='a', index=False)
    print(f"  - Appended results to: {file_path}")

In [ ]:
# ==============================================================================
# 12A. UNIVERSAL BAYESIAN FITTER (SUBPROCESS - FORMATTING FIXED)
# ==============================================================================
import subprocess
import os
import pandas as pd
import numpy as np

def run_brms_subprocess(model_formula, model_name, output_file, model_description, df_to_use, family="cumulative('logit')", cores=4):
    print(f"\n----- Running Bayesian Analysis: {model_name} (Subprocess) -----")
    
    # Define Paths
    rds_path = os.path.join(MODEL_OUTPUT_FOLDER, f"{model_name}.rds")
    summary_path = os.path.join(MODEL_OUTPUT_FOLDER, f"{model_name}_summary.txt")
    temp_data_path = os.path.join(MODEL_OUTPUT_FOLDER, f"temp_data_{model_name}.csv")
    temp_script_path = os.path.join(MODEL_OUTPUT_FOLDER, f"temp_script_{model_name}.R")
    temp_coeffs_path = os.path.join(MODEL_OUTPUT_FOLDER, f"temp_coeffs_{model_name}.csv")
    
    # Save Data
    print(f"  > Saving temporary data: {temp_data_path}")
    df_to_use.to_csv(temp_data_path, index=False)
    
    # R Script Content
    r_content = f"""
    suppressPackageStartupMessages(library(brms))
    
    data <- read.csv("{temp_data_path}")
    
    # Ensure Factors
    if("Matteson_Stress" %in% names(data)) {{
        data$Matteson_Stress <- factor(data$Matteson_Stress,
            levels = c('Unstressed', 'Secondary_stress', 'Primary_stress'), ordered = TRUE)
    }}
    
    # Priors
    priors <- c(set_prior("normal(0, 10)", class = "b"))
    if(grepl("|", "{model_formula}", fixed=TRUE)) {{
        priors <- c(priors, set_prior("student_t(3, 0, 10)", class = "sd"))
    }}
    
    # Fit Model
    cat("  > Starting Brms Fit (Cores: {cores})...\\n")
    model <- brm(
        formula = as.formula({model_formula}),
        data = data,
        family = {family},
        prior = priors,
        iter = 2000, warmup = 1000, chains = 4, 
        cores = {cores}, 
        file = "{rds_path}",
        file_refit = "on_change"
    )
    
    # --- EXTRACT RESULTS (FIXED FORMATTING) ---
    cat("  > Extracting Results...\\n")
    
    # 1. Generate Summary Object
    model_summary <- summary(model)
    
    # 2. Save Text Report (Force 4 digits)
    capture.output(print(model_summary, digits = 4), file = "{summary_path}")
    
    # 3. Extract Coefficients Table (This includes Rhat and ESS)
    # We switch from fixef() to model_summary$fixed to get the diagnostics
    coeffs <- as.data.frame(model_summary$fixed)
    
    # Add Term Names
    coeffs <- cbind(Term = rownames(coeffs), coeffs)
    
    # Round Numeric Columns to 4 decimals
    nums <- sapply(coeffs, is.numeric)
    coeffs[nums] <- round(coeffs[nums], 4)
    
    # Save to CSV
    write.csv(coeffs, file = "{temp_coeffs_path}", row.names=FALSE)
    """
    
    with open(temp_script_path, "w") as f: f.write(r_content)
        
    try:
        print(f"  > Executing R in background...")
        subprocess.run(["Rscript", temp_script_path], check=True, capture_output=True)
        
        if os.path.exists(temp_coeffs_path):
            # Read back as object to preserve the R-side formatting
            coeffs_df = pd.read_csv(temp_coeffs_path).astype(object)
            coeffs_df.fillna('-', inplace=True)
            append_results_to_csv(output_file, f"--- {model_description} ---", coeffs_df)
            print(f"  > Success! Results saved.")
            
            # Cleanup
            if os.path.exists(temp_data_path): os.remove(temp_data_path)
            if os.path.exists(temp_script_path): os.remove(temp_script_path)
            if os.path.exists(temp_coeffs_path): os.remove(temp_coeffs_path)
        else:
            print("!!! Error: No coefficients generated.")
    except subprocess.CalledProcessError as e:
        print(f"!!! R Execution Failed !!!\n{e.stderr.decode()}")

In [ ]:
# ==============================================================================
# 12B. BAYESIAN COMPARISON (MEMORY-SAFE SUBPROCESS VERSION)
# ==============================================================================
import subprocess
import os

def compare_brms_models_subprocess(model_name_1, model_name_2):
    print(f"\n----- Comparing: {model_name_1} vs {model_name_2} (Memory-Safe) -----")
    
    # Unique temp script path to allow parallel notebook execution
    temp_script_path = os.path.join(MODEL_OUTPUT_FOLDER, f"temp_compare_{model_name_1}_vs_{model_name_2}.R")
    
    # We explicitly remove models (rm) and run garbage collection (gc) between steps
    r_content = f"""
    suppressPackageStartupMessages(library(brms))
    suppressPackageStartupMessages(library(loo))
    
    # Allow R to use as much RAM as needed for the large objects
    options(future.globals.maxSize = 16 * 1024^3)
    
    path1 <- file.path("{MODEL_OUTPUT_FOLDER}", paste0("{model_name_1}", ".rds"))
    path2 <- file.path("{MODEL_OUTPUT_FOLDER}", paste0("{model_name_2}", ".rds"))
    
    if(!file.exists(path1) || !file.exists(path2)) {{
        stop("One or both model files (.rds) do not exist.")
    }}
    
    # --- STEP 1: Process Model 1 ---
    cat(paste0("  > Loading ", "{model_name_1}", "...\\n"))
    m1 <- readRDS(path1)
    
    cat("  > Calculating LOO 1 (Full Posterior)...\\n")
    # cores=1 prevents memory duplication during matrix calculation
    # reloo=FALSE prevents expensive refitting on outliers
    l1 <- loo(m1, reloo = FALSE, cores = 1)
    
    # CRITICAL: Remove Model 1 from RAM before loading Model 2 to free ~8GB RAM
    rm(m1)
    gc() 
    
    # --- STEP 2: Process Model 2 ---
    cat(paste0("  > Loading ", "{model_name_2}", "...\\n"))
    m2 <- readRDS(path2)
    
    cat("  > Calculating LOO 2 (Full Posterior)...\\n")
    l2 <- loo(m2, reloo = FALSE, cores = 1)
    
    # Remove Model 2
    rm(m2)
    gc()
    
    # --- STEP 3: Compare ---
    cat("  > Performing Comparison...\\n")
    print(loo_compare(l1, l2))
    """
    
    with open(temp_script_path, "w") as f:
        f.write(r_content)
        
    try:
        result = subprocess.run(
            ["Rscript", temp_script_path], 
            check=True, 
            capture_output=True, 
            text=True
        )
        print(result.stdout)
        
        if os.path.exists(temp_script_path):
            os.remove(temp_script_path)
            
    except subprocess.CalledProcessError as e:
        print("!!! Comparison Failed !!!")
        print(e.stderr)

## Part 4: Interpretation and Validation
Once the best model is selected, we interpret the results visually:
*   **Conditional Effects Plots:** Visualizing significant interactions (e.g., how Duration effects change based on Syllable Position).
*   **Posterior Predictive Checks (PPC):** Validating the model fit by comparing the model's simulated predictions against the actual observed data distribution.

In [ ]:
# ==============================================================================
# 12C. BAYESIAN ANALYSIS: VISUALIZATION FUNCTION (SUBPROCESS VERSION)
# ==============================================================================
import subprocess
import os

def plot_conditional_effects_subprocess(model_name, effect_vars, plot_filename):
    print(f"\n----- Generating Plot & Data: '{effect_vars}' (Subprocess) -----")

    # Define Paths
    temp_script_path = os.path.join(MODEL_OUTPUT_FOLDER, "temp_plot_script.R")
    csv_filename = plot_filename.replace(".png", "_Data.csv")
    
    # R Script Content
    r_content = f"""
    library(brms)
    library(ggplot2)

    # Paths
    model_path <- file.path("{MODEL_OUTPUT_FOLDER}", paste0("{model_name}", ".rds"))
    plot_path <- file.path("{MODEL_OUTPUT_FOLDER}", "{plot_filename}")
    csv_path <- file.path("{MODEL_OUTPUT_FOLDER}", "{csv_filename}")

    cat("  > Loading model...\\n")
    if (!file.exists(model_path)) {{
        stop(paste("Model file not found:", model_path))
    }}
    model <- readRDS(model_path)

    eff_string <- "{effect_vars}"
    cat(paste("  > Processing effect:", eff_string, "\\n"))

    # Handle Interaction vs Main Effect
    if (grepl(":", eff_string)) {{
        parts <- unlist(strsplit(eff_string, ":"))
        pred_x <- parts[1]
        pred_cond <- parts[2]
        
        # Create conditions for the second variable
        cond_data <- model$data[[pred_cond]]
        if (is.numeric(cond_data)) {{
            conds <- data.frame(var = c(-1, 0, 1))
            names(conds) <- pred_cond
        }} else {{
            conds <- data.frame(var = unique(cond_data))
            names(conds) <- pred_cond
        }}
        
        effects_data <- conditional_effects(model, effects = pred_x, conditions = conds, categorical = TRUE)
    }} else {{
        effects_data <- conditional_effects(model, effects = eff_string, categorical = TRUE)
    }}

    # Save Data
    raw_plot_data <- effects_data[[1]]
    write.csv(raw_plot_data, csv_path, row.names = FALSE)
    cat(paste("  > Saved Data:", csv_path, "\\n"))

    # Save Plot
    plot_list <- plot(effects_data, plot = FALSE)
    ggsave(plot_path, plot = plot_list[[1]], width = 10, height = 6, dpi = 300)
    cat(paste("  > Saved Plot:", "{plot_filename}", "\\n"))
    """

    # Write and Run
    with open(temp_script_path, "w") as f:
        f.write(r_content)

    try:
        subprocess.run(["Rscript", temp_script_path], check=True, capture_output=True)
        print(f"  > Success! Plot saved to {plot_filename}")
    except subprocess.CalledProcessError as e:
        print(f"!!! Visualization Failed !!!\n{e.stderr.decode()}")

In [ ]:
# ==============================================================================
# 12D. BAYESIAN ANALYSIS: PPC FUNCTION (SUBPROCESS VERSION)
# ==============================================================================
def run_ppc_subprocess(model_name, plot_filename):
    print(f"\n----- Running PPC: '{model_name}' (Subprocess) -----")

    temp_script_path = os.path.join(MODEL_OUTPUT_FOLDER, "temp_ppc_script.R")
    
    r_content = f"""
    library(brms)
    library(ggplot2)

    model_path <- file.path("{MODEL_OUTPUT_FOLDER}", paste0("{model_name}", ".rds"))
    plot_path <- file.path("{MODEL_OUTPUT_FOLDER}", "{plot_filename}")

    cat("  > Loading model...\\n")
    if (!file.exists(model_path)) {{
        stop("Model file not found.")
    }}
    model <- readRDS(model_path)

    cat("  > Generating Bar Plot (Ordinal)...\\n")
    # ndraws=100 is usually sufficient for a quick check
    ppc_plot <- pp_check(model, type = "bars", ndraws = 100)

    ggsave(plot_path, plot = ppc_plot, width = 8, height = 6, dpi = 300)
    cat(paste("  > Saved Plot:", "{plot_filename}", "\\n"))
    """

    with open(temp_script_path, "w") as f:
        f.write(r_content)

    try:
        subprocess.run(["Rscript", temp_script_path], check=True, capture_output=True)
        print(f"  > Success! PPC Plot saved to {plot_filename}")
    except subprocess.CalledProcessError as e:
        print(f"!!! PPC Failed !!!\n{e.stderr.decode()}")

## Part 5: Exploratory Analysis (Conditional Random Forests)
This phase uses Machine Learning to provide a data-driven ranking of "Cue Importance."

**Methodology:**
We use **Conditional Inference Forests (`cforest`)** rather than standard Random Forests.
*   **Why?** Standard RFs are biased towards variables with many categories. `cforest` uses unbiased statistical tests for splitting.
*   **Importance Metric:** We calculate **Conditional Permutation Importance**. This mathematically corrects for correlations between predictors (e.g., Duration vs. Pitch), ensuring that a cue is only ranked high if it provides *unique* information.

**Models Run:**
1.  **Full Model:** Includes all acoustic AND structural predictors (Position, Index).
2.  **Acoustic-Only Model:** Includes ONLY acoustic cues (Pitch, Duration, Intensity, Formants) to see the hierarchy of phonetic realization without structural confounds.

In [ ]:
# ==============================================================================
# 12E. RANDOM FOREST (SUBPROCESS - PERMIMP / LEGACY PARTY)
# ==============================================================================
import subprocess
import os
import pandas as pd

def run_cforest_subprocess(output_file, model_label, predictors_list, df_to_use, mtry_val=5, ntree_val=1000):
    print(f"\n----- Running RF (permimp): {model_label}, ntree={ntree_val}, mtry={mtry_val} -----")
    
    # 1. Define Unique Paths
    temp_data = os.path.join(MODEL_OUTPUT_FOLDER, f"temp_rf_{model_label}.csv")
    temp_script = os.path.join(MODEL_OUTPUT_FOLDER, f"temp_rf_{model_label}.R")
    temp_imp = os.path.join(MODEL_OUTPUT_FOLDER, f"temp_rf_imp_{model_label}.csv")
    report_path = os.path.join(MODEL_OUTPUT_FOLDER, f"RF_Report_{model_label}.txt")
    plot_path = os.path.join(MODEL_OUTPUT_FOLDER, f"RF_Plot_{model_label}.png")
    
    # 2. Save Data
    df_to_use.to_csv(temp_data, index=False)
    
    # 3. R Script
    pred_vec = "c('" + "', '".join(predictors_list) + "')"
    
    r_content = f"""
    # Load Legacy Libraries
    suppressPackageStartupMessages(library(party))
    suppressPackageStartupMessages(library(permimp))
    suppressPackageStartupMessages(library(caret))
    suppressPackageStartupMessages(library(ggplot2))
    suppressPackageStartupMessages(library(pROC))
    
    # Load Data
    df <- read.csv("{temp_data}")
    
    # --- DATA PREPARATION ---
    # Convert Target to Factor
    df$Matteson_Stress <- factor(df$Matteson_Stress)
    possible_levels <- c('Unstressed', 'Secondary_stress', 'Primary_stress')
    actual_levels <- intersect(possible_levels, levels(df$Matteson_Stress))
    if(length(actual_levels) > 0) {{
         df$Matteson_Stress <- factor(df$Matteson_Stress, levels = actual_levels, ordered = TRUE)
    }}

    # Convert Predictors to Factors
    cat_vars <- c('Phrase_Position', 'Is_Interrogative', 'Is_Exclamative',
                  'Preceded_by_Silence', 'Followed_by_Silence', 'Vowel')
    for (var in cat_vars) {{
        if (var %in% names(df)) {{ df[[var]] <- as.factor(df[[var]]) }}
    }}
    
    # --- SANITY CHECK: Drop Constant Predictors ---
    requested_predictors <- {pred_vec}
    valid_predictors <- c()
    for (v in requested_predictors) {{
        if (v %in% names(df)) {{
            if (length(unique(df[[v]])) > 1) {{
                valid_predictors <- c(valid_predictors, v)
            }} else {{
                cat(paste0("    ! WARNING: Dropping constant predictor '", v, "'\\n"))
            }}
        }}
    }}
    df <- df[, c("Matteson_Stress", valid_predictors)]
    
    # --- MODEL FITTING (Legacy cforest) ---
    cat(paste0("  > Fitting cforest (legacy)...\\n"))
    set.seed(42)
    
    # cforest_unbiased is the built-in function in 'party' that handles 
    # unbiased controls (quad test statistic, subsampling, etc.)
    rf <- cforest(Matteson_Stress ~ ., 
                  data = df, 
                  controls = cforest_unbiased(ntree = {ntree_val}, mtry = {mtry_val}))
    
    # --- VARIABLE IMPORTANCE (permimp) ---
    cat("  > Calculating Conditional Importance via permimp...\\n")
    
    # threshold=0.2 includes predictors with p-value < 0.8 in the conditioning grid.
    # This is aggressive conditioning suited for high collinearity.
    imp_obj <- permimp(rf, conditional = TRUE, threshold = 0.2, asParty = TRUE)
    
    # Extract values
    imp_values <- imp_obj$values
    write.csv(data.frame(Feature=names(imp_values), Importance=as.numeric(imp_values)), "{temp_imp}", row.names=FALSE)
    
    # Plot
    p <- ggplot(data.frame(F=names(imp_values), I=as.numeric(imp_values)), aes(x=I, y=reorder(F, I))) + 
         geom_point(size=3) + theme_bw() + 
         labs(title=paste0("RF Importance: {model_label}"), x="Conditional Importance (permimp)", y="Feature") +
         geom_vline(xintercept = 0, linetype="dashed", color="red")
    ggsave("{plot_path}", plot=p, width=8, height=6)
    
    # --- REPORTING ---
    cat("  > Generating Report...\\n")
    sink("{report_path}")
    
    # Predictions in 'party' work differently than 'partykit'
    # OOB = TRUE is default for predict in 'party'
    preds_class <- predict(rf, OOB = TRUE, type = "response")
    print(confusionMatrix(preds_class, df$Matteson_Stress, mode = "everything"))
    
    tryCatch({{
        # Probability extraction for 'party' objects
        preds_prob_list <- predict(rf, OOB = TRUE, type = "prob")
        preds_prob <- do.call(rbind, preds_prob_list)
        colnames(preds_prob) <- levels(df$Matteson_Stress)
        
        if (nlevels(df$Matteson_Stress) == 2) {{
            target_col <- levels(df$Matteson_Stress)[2]
            roc_obj <- roc(df$Matteson_Stress, preds_prob[, target_col], quiet=TRUE)
            cat(paste0("\\nBinary AUC: ", round(as.numeric(auc(roc_obj)), 4), "\\n"))
        }} else {{
            roc_obj <- multiclass.roc(df$Matteson_Stress, preds_prob, quiet=TRUE)
            cat(paste0("\\nMulti-class AUC: ", round(as.numeric(auc(roc_obj)), 4), "\\n"))
        }}
    }}, error = function(e) {{ cat(paste0("\\nAUC Error: ", e$message, "\\n")) }})
    
    sink()
    """
    
    with open(temp_script, "w") as f: f.write(r_content)
    
    try:
        subprocess.run(["Rscript", temp_script], check=True, capture_output=True)
        
        if os.path.exists(temp_imp):
            imp_df = pd.read_csv(temp_imp)
            append_results_to_csv(output_file, f"--- RF Results: {model_label} ---", imp_df)
            print(f"  > RF Complete. Report saved to {report_path}")
            
            # Cleanup
            if os.path.exists(temp_data): os.remove(temp_data)
            if os.path.exists(temp_script): os.remove(temp_script)
            if os.path.exists(temp_imp): os.remove(temp_imp)
        else:
            print("!!! Error: No importance file generated.")
            
    except subprocess.CalledProcessError as e:
        print(f"!!! RF Failed !!!\n{e.stderr.decode()}")

## Execution Blocks for Bayesian Models

In [ ]:
# ==============================================================================
# 13. INITIALIZE OUTPUT FILES (OPTIONAL)
# ==============================================================================
# Run this ONLY if you want to delete old results and start fresh.
# if __name__ == '__main__':
    # Uncomment to clear files
    # for f in [BRM_OUTPUT_FILE, RF_FULL_OUTPUT_FILE, RF_ACOUSTIC_OUTPUT_FILE, RF_BINARY_A_OUTPUT_FILE, RF_BINARY_B_OUTPUT_FILE]:
    #     if os.path.exists(f):
    #         os.remove(f)
    #         print(f"Deleted: {f}")
#     pass

In [ ]:
# ==============================================================================
# 14. EXECUTE ITERATIVE MODEL BUILDING: STEP 1 (MODEL 0) - SUBPROCESS VERSION
# ==============================================================================

if __name__ == '__main__':
    if  'main_df' in locals() and main_df is not None:
        try:
            # Define the formula for Model 0 (Fixed Effects ONLY)
            fixed_effects_formula = """
                Duration_z + Pitch_z + Intensity_Q2_z + Spectral_Tilt_DB_z +
                F1_z + F2_z + Pace_SPS_z + Syllable_position_z + Syllable_Index_z +
                Phrase_Position + Is_Interrogative + Is_Exclamative +
                Preceded_by_Silence + Followed_by_Silence + Vowel
            """

            model_0_formula = f"bf(Matteson_Stress ~ {fixed_effects_formula})"

            # Run and Save Model 0 using the NEW Subprocess function
            # Note: We now pass 'main_df' explicitly as the last argument
            run_brms_subprocess(
                model_formula=model_0_formula,
                model_name="Model_0_Baseline",
                output_file=BRM_OUTPUT_FILE,
                model_description="MODEL 0: BASELINE (FIXED EFFECTS ONLY)",
                df_to_use=main_df 
            )

            print("\n--- Model 0 execution complete. ---")

        except Exception as e:
            print(f"\n!!! An unexpected error occurred during the main execution: {e} !!!")
            import traceback
            traceback.print_exc()
    else:
        print("!!! ERROR: 'main_df' not found. Please run the data loading cells first. !!!")

In [ ]:
# ==============================================================================
# 15. EXECUTE ITERATIVE MODEL BUILDING: STEP 2 (MODEL 1a) (10,000 data points -> 1 h 40 mins)
# ==============================================================================

if __name__ == '__main__':
    if 'main_df' in locals() and main_df is not None:
        fixed_effects_formula = """
            Duration_z + Pitch_z + Intensity_Q2_z + Spectral_Tilt_DB_z +
            F1_z + F2_z + Pace_SPS_z + Syllable_position_z + Syllable_Index_z +
            Phrase_Position + Is_Interrogative + Is_Exclamative +
            Preceded_by_Silence + Followed_by_Silence + Vowel
        """
        random_effects_intercepts = "(1|Speaker) + (1|Words)"
        model_1a_formula = f"bf(Matteson_Stress ~ {fixed_effects_formula} + {random_effects_intercepts})"

        run_brms_subprocess(
            model_formula=model_1a_formula,
            model_name="Model_1a_Random_Intercepts",
            output_file=BRM_OUTPUT_FILE,
            model_description="MODEL 1a: ADDING RANDOM INTERCEPTS",
            df_to_use=main_df
        )

In [ ]:
# ==============================================================================
# 16. EXECUTE ITERATIVE MODEL BUILDING: STEP 3 (COMPARE MODEL 0 vs MODEL 1a)
# ==============================================================================

if __name__ == '__main__':
    # Define the names of the two models we want to compare.
    # These must match the 'model_name' arguments from the previous execution cells.
    model_to_compare_1 = "Model_0_Baseline"
    model_to_compare_2 = "Model_1a_Random_Intercepts"

    # Call the comparison function.
    compare_brms_models_subprocess(model_to_compare_1, model_to_compare_2)

In [ ]:
# ==============================================================================
# 17. EXECUTE ITERATIVE MODEL BUILDING: STEP 4 (MODEL 1b)
# ==============================================================================

if __name__ == '__main__':
    if 'main_df' in locals() and main_df is not None:
        try:
            # --- 1. Define the formula for Model 1 (Add Random Intercepts) ---
            fixed_effects_formula = """
                Duration_z + Pitch_z + Intensity_Q2_z + Spectral_Tilt_DB_z +
                F1_z + F2_z + Pace_SPS_z + Syllable_position_z + Syllable_Index_z +
                Phrase_Position + Is_Interrogative + Is_Exclamative +
                Preceded_by_Silence + Followed_by_Silence + Vowel
            """

            # As planned, we are excluding Utterance_id to ensure model stability.
            random_effects_intercepts = """
                (1|Speaker) + (1|Words) + (1|Utterance_id)
            """

            model_1b_formula = f"bf(Matteson_Stress ~ {fixed_effects_formula} + {random_effects_intercepts})"

            # --- 2. Run and Save Model 1 ---
            # Call the new, dedicated function for mixed-effects models.
            run_brms_subprocess(
                model_formula=model_1b_formula,
                model_name="Model_1b_Random_Intercepts",
                output_file=BRM_OUTPUT_FILE,
                model_description="MODEL 1b: ADDING RANDOM INTERCEPTS: + Utterance",
                df_to_use=main_df
            )

            print("\n--- Model 1b execution complete. ---")

        except Exception as e:
            print(f"\n!!! An unexpected error occurred during Model 1 execution: {e} !!!")
            import traceback
            traceback.print_exc()
    else:
        print("!!! ERROR: 'main_df' not found. Please run the data loading cells first. !!!")

In [ ]:
# ==============================================================================
# 18. EXECUTE ITERATIVE MODEL BUILDING: STEP 5 (COMPARE MODEL 1a vs MODEL 1b)
# ==============================================================================

if __name__ == '__main__':
    # Define the names of the two models we want to compare.
    # These must match the 'model_name' arguments from the previous execution cells.
    model_to_compare_1 = "Model_1a_Random_Intercepts"
    model_to_compare_2 = "Model_1b_Random_Intercepts"

    # Call the comparison function.
    compare_brms_models_subprocess(model_to_compare_1, model_to_compare_2)

In [ ]:
# ==============================================================================
# 19. EXECUTE ITERATIVE MODEL BUILDING: STEP 6 (MODEL 1c)
# ==============================================================================

if __name__ == '__main__':
    if 'main_df' in locals() and main_df is not None:
        try:
            # --- 1. Define the formula for Model 1 (Add Random Intercepts) ---
            fixed_effects_formula = """
                Duration_z + Pitch_z + Intensity_Q2_z + Spectral_Tilt_DB_z +
                F1_z + F2_z + Pace_SPS_z + Syllable_position_z + Syllable_Index_z +
                Phrase_Position + Is_Interrogative + Is_Exclamative +
                Preceded_by_Silence + Followed_by_Silence + Vowel
            """

            # As planned, we are excluding Utterance_id to ensure model stability.
            random_effects_intercepts = """
                (1|File_Name) + (1|Speaker) + (1|Words)
            """

            model_1c_formula = f"bf(Matteson_Stress ~ {fixed_effects_formula} + {random_effects_intercepts})"

            # --- 2. Run and Save Model 1 ---
            # Call the new, dedicated function for mixed-effects models.
            run_brms_subprocess(
                model_formula=model_1c_formula,
                model_name="Model_1c_Random_Intercepts",
                output_file=BRM_OUTPUT_FILE,
                model_description="MODEL 1c: ADDING RANDOM INTERCEPTS: + File",
                df_to_use=main_df
            )

            print("\n--- Model 1c execution complete. ---")

        except Exception as e:
            print(f"\n!!! An unexpected error occurred during Model 1 execution: {e} !!!")
            import traceback; traceback.print_exc()
    else:
        print("!!! ERROR: 'main_df' not found. Please run the data loading cells first. !!!")

In [ ]:
# ==============================================================================
# 20. EXECUTE ITERATIVE MODEL BUILDING: STEP 7 (COMPARE MODEL 1a vs MODEL 1c)
# ==============================================================================

if __name__ == '__main__':
    # Define the names of the two models we want to compare.
    # These must match the 'model_name' arguments from the previous execution cells.
    model_to_compare_1 = "Model_1a_Random_Intercepts"
    model_to_compare_2 = "Model_1c_Random_Intercepts"

    # Call the comparison function.
    compare_brms_models_subprocess(model_to_compare_1, model_to_compare_2)

In [ ]:
# ==============================================================================
# 21. EXECUTE ITERATIVE MODEL BUILDING: STEP 8 (MODEL 2)
# ==============================================================================

if __name__ == '__main__':
    if 'main_df' in locals() and main_df is not None:
        try:
            # --- 1. Define the formula for Model 2 (Add Random Slopes) ---
            fixed_effects_formula = """
                Duration_z + Pitch_z + Intensity_Q2_z + Spectral_Tilt_DB_z +
                F1_z + F2_z + Pace_SPS_z + Syllable_position_z + Syllable_Index_z +
                Phrase_Position + Is_Interrogative + Is_Exclamative +
                Preceded_by_Silence + Followed_by_Silence + Vowel
            """

            # This is the same random intercepts structure from Model 1.
            random_effects_intercepts = """
                (1|Words)
            """

            # Define the Random Slopes for Speaker
            # This term allows the effect of the main acoustic cues to vary by Speaker.
            # It models both a random intercept for Speaker and random slopes for the cues.
            random_slopes_for_speaker = """
                (1 + Duration_z + Pitch_z + Spectral_Tilt_DB_z | Speaker)
            """

            model_2_formula = f"bf(Matteson_Stress ~ {fixed_effects_formula} + {random_effects_intercepts} + {random_slopes_for_speaker})"

            # --- 2. Run and Save Model 2 ---
            run_brms_subprocess(
                model_formula=model_2_formula,
                model_name="Model_2_Random_Slopes",
                output_file=BRM_OUTPUT_FILE,
                model_description="MODEL 2: ADDING RANDOM SLOPES BY SPEAKER",
                df_to_use=main_df
            )

            print("\n--- Model 2 execution complete. ---")

        except Exception as e:
            print(f"\n!!! An unexpected error occurred during Model 2 execution: {e} !!!")
            import traceback; traceback.print_exc()
    else:
        print("!!! ERROR: 'main_df' not found. Please run the data loading cells first. !!!")

In [ ]:
# ==============================================================================
# 22. EXECUTE ITERATIVE MODEL BUILDING: STEP 9 (COMPARE MODEL 1a vs MODEL 2)
# ==============================================================================

if __name__ == '__main__':
    # Define the names of the two models we want to compare.
    model_to_compare_1 = "Model_1a_Random_Intercepts"
    model_to_compare_2 = "Model_2_Random_Slopes"

    # Call the comparison function.
    compare_brms_models_subprocess(model_to_compare_1, model_to_compare_2)

In [ ]:
# ==============================================================================
# 23. EXECUTE ITERATIVE MODEL BUILDING: STEP 10 (MODEL 3a - Vowel Quality Interactions)
# ==============================================================================

if __name__ == '__main__':
    if 'main_df' in locals() and main_df is not None:
        try:
            # --- 1. Define the formula for Model 3a ---
            # Base fixed effects
            fixed_effects_formula = """
                Duration_z + Pitch_z + Intensity_Q2_z + Spectral_Tilt_DB_z +
                F1_z + F2_z + Pace_SPS_z + Syllable_position_z + Syllable_Index_z +
                Phrase_Position + Is_Interrogative + Is_Exclamative +
                Preceded_by_Silence + Followed_by_Silence + Vowel
            """

            # Base random effects from Model 2
            random_effects_formula = """
                (1|Words) +
                (1 + Duration_z + Pitch_z + Spectral_Tilt_DB_z | Speaker)
            """

            # Define Vowel Quality Interactions ===
            vowel_quality = """
                F1_z * Vowel
            """

            model_3a_formula = (f"bf(Matteson_Stress ~ {fixed_effects_formula} + "
                              f"{vowel_quality} + "
                              f"{random_effects_formula})")

            # --- 2. Run and Save Model 3a ---
            run_brms_subprocess(
                model_formula=model_3a_formula,
                model_name="Model_3a_Confounding_Vowel_Quality",
                output_file=BRM_OUTPUT_FILE,
                model_description="MODEL 3a: ADDING CONFOUNDING VOWEL QUALITY INTERACTIONS",
                df_to_use=main_df
            )

            print("\n--- Model 3a execution complete. ---")

        except Exception as e:
            print(f"\n!!! An unexpected error occurred during Model 3a execution: {e} !!!")
            import traceback; traceback.print_exc()
    else:
        print("!!! ERROR: Prerequisite steps not complete. Please run the required setup cells first. !!!")

In [ ]:
# ==============================================================================
# 24. EXECUTE ITERATIVE MODEL BUILDING: STEP 11 (COMPARE MODEL 2 vs MODEL 3a)
# ==============================================================================

if __name__ == '__main__':
    # Define the names of the two models we want to compare.
    # We are comparing our previous best model (Model 2) against the new one.
    model_to_compare_1 = "Model_2_Random_Slopes"
    model_to_compare_2 = "Model_3a_Confounding_Vowel_Quality"

    # Call the comparison function.
    compare_brms_models_subprocess(model_to_compare_1, model_to_compare_2)

In [ ]:
# ==============================================================================
# 25. EXECUTE ITERATIVE MODEL BUILDING: STEP 12 (MODEL 3b - Intonational Interactions)
# ==============================================================================

if __name__ == '__main__':
    if 'main_df' in locals() and main_df is not None:
        try:
            # --- 1. Define the formula for Model 3b ---
            # Base fixed effects
            fixed_effects_formula = """
                Duration_z + Pitch_z + Intensity_Q2_z + Spectral_Tilt_DB_z +
                F1_z + F2_z + Pace_SPS_z + Syllable_position_z + Syllable_Index_z +
                Phrase_Position + Is_Interrogative + Is_Exclamative +
                Preceded_by_Silence + Followed_by_Silence + Vowel
            """

            # Base random effects from Model 2
            random_effects_formula = """
                (1|Words) +
                (1 + Duration_z + Pitch_z + Spectral_Tilt_DB_z | Speaker)
            """

            # Define Vowel Quality Interactions
            vowel_quality = """
                F1_z * Vowel
            """

            # Define the Intonational Interaction Terms
            intonational_interactions = """
                Duration_z * Is_Exclamative +
                Pitch_z * Is_Exclamative +
                Spectral_Tilt_DB_z * Is_Exclamative +
                Duration_z * Is_Interrogative +
                Pitch_z * Is_Interrogative +
                Spectral_Tilt_DB_z * Is_Interrogative
            """

            model_3b_formula = (f"bf(Matteson_Stress ~ {fixed_effects_formula} + "
                              f"{vowel_quality} + "
                              f"{intonational_interactions} + "
                              f"{random_effects_formula})")

            # --- 2. Run and Save Model 3b ---
            run_brms_subprocess(
                model_formula=model_3b_formula,
                model_name="Model_3b_Intonational_Interactions",
                output_file=BRM_OUTPUT_FILE,
                model_description="MODEL 3b: ADDING INTONATIONAL INTERACTIONS",
                df_to_use=main_df
            )

            print("\n--- Model 3b execution complete. ---")

        except Exception as e:
            print(f"\n!!! An unexpected error occurred during Model 3b execution: {e} !!!")
            import traceback; traceback.print_exc()
    else:
        print("!!! ERROR: Prerequisite steps not complete. Please run the required setup cells first. !!!")

In [ ]:
# ==============================================================================
# 26. EXECUTE ITERATIVE MODEL BUILDING: STEP 13 (COMPARE MODEL 3a vs MODEL 3b)
# ==============================================================================

if __name__ == '__main__':
    # Define the names of the two models we want to compare.
    # We are comparing our previous best model (Model 2) against the new one.
    model_to_compare_1 = "Model_3a_Confounding_Vowel_Quality"
    model_to_compare_2 = "Model_3b_Intonational_Interactions"

    # Call the comparison function.
    compare_brms_models_subprocess(model_to_compare_1, model_to_compare_2)

In [ ]:
# ==============================================================================
# 27. EXECUTE ITERATIVE MODEL BUILDING: STEP 14 (MODEL 3c - Cue Trading)
# ==============================================================================

if __name__ == '__main__':
    if 'main_df' in locals() and main_df is not None:
        try:
            # --- 1. Define the formula for Model 3c ---
            # Base fixed effects
            fixed_effects_formula = """
                Duration_z + Pitch_z + Intensity_Q2_z + Spectral_Tilt_DB_z +
                F1_z + F2_z + Pace_SPS_z + Syllable_position_z + Syllable_Index_z +
                Phrase_Position + Is_Interrogative + Is_Exclamative +
                Preceded_by_Silence + Followed_by_Silence + Vowel
            """

            # Base random effects from Model 2
            random_effects_formula = """
                (1|Words) +
                (1 + Duration_z + Pitch_z + Spectral_Tilt_DB_z | Speaker)
            """

            # Define Vowel Quality Interactions
            vowel_quality = """
                F1_z * Vowel
            """

            # Define the Cue Trading and Vowel Quality Interactions
            cue_trading_interactions = """
                Duration_z * Pitch_z * Spectral_Tilt_DB_z
            """

            model_3c_formula = (f"bf(Matteson_Stress ~ {fixed_effects_formula} + "
                              f"{vowel_quality} + "
                              f"{cue_trading_interactions} + "
                              f"{random_effects_formula})")

            # --- 2. Run and Save Model 3c ---
            run_brms_subprocess(
                model_formula=model_3c_formula,
                model_name="Model_3c_CueTrading_Interactions",
                output_file=BRM_OUTPUT_FILE,
                model_description="MODEL 3c: ADDING CUE TRADING INTERACTIONS",
                df_to_use=main_df
            )

            print("\n--- Model 3c execution complete. ---")

        except Exception as e:
            print(f"\n!!! An unexpected error occurred during Model 3c execution: {e} !!!")
            import traceback; traceback.print_exc()
    else:
        print("!!! ERROR: Prerequisite steps not complete. Please run the required setup cells first. !!!")

In [ ]:
# ==============================================================================
# 28. EXECUTE ITERATIVE MODEL BUILDING: STEP 15 (COMPARE MODEL 3b vs MODEL 3c)
# ==============================================================================

if __name__ == '__main__':
    # We are comparing our previous mixed model (Model 3b) against the new one.
    model_to_compare_1 = "Model_3a_Confounding_Vowel_Quality"
    model_to_compare_2 = "Model_3c_CueTrading_Interactions"

    compare_brms_models_subprocess(model_to_compare_1, model_to_compare_2)

In [ ]:
"""THIS IS THE FINAL MODEL. ¡RUN THIS!"""
# ==============================================================================
# 29. EXECUTE ITERATIVE MODEL BUILDING: STEP 16 (MODEL 3d - FINAL MODEL)
# ==============================================================================

if __name__ == '__main__':
    if 'main_df' in locals() and main_df is not None:
        try:
            # --- 1. Define the formula for Model 3d ---
            # This formula builds directly on the structure of Model 2.

            # Base fixed effects
            fixed_effects_formula = """
                Duration_z + Pitch_z + Intensity_Q2_z + Spectral_Tilt_DB_z +
                F1_z + F2_z + Pace_SPS_z + Syllable_position_z + Syllable_Index_z +
                Phrase_Position + Is_Interrogative + Is_Exclamative +
                Preceded_by_Silence + Followed_by_Silence + Vowel
            """

            # Base random effects from Model 2
            random_effects_intercepts = "(1|Words)"
            random_slopes_for_speaker = "(1 + Duration_z + Pitch_z + Spectral_Tilt_DB_z | Speaker)"

            # Define Vowel Quality Interactions
            vowel_quality = """
                F1_z * Vowel
            """

            # Define the Cue Trading and Vowel Quality Interactions
            cue_trading_interactions = """
                Duration_z * Pitch_z * Spectral_Tilt_DB_z
            """

            # Define the Positional Interaction Terms
            positional_interactions = """
                Duration_z * Syllable_position_z +
                Spectral_Tilt_DB_z * Syllable_position_z
            """

            # Combine all parts into the final formula for brms
            model_3d_formula = (f"bf(Matteson_Stress ~ {fixed_effects_formula} + "
                              f"{vowel_quality} + "
                              f"{cue_trading_interactions} + "
                              f"{positional_interactions} + "
                              f"{random_effects_intercepts} + {random_slopes_for_speaker})")

            # --- 2. Run and Save Model 4 ---
            run_brms_subprocess(
                model_formula=model_3d_formula,
                model_name="Model_3d_Syllable_Position_Interactions",
                output_file=BRM_OUTPUT_FILE,
                model_description="MODEL 3d: ADDING SYLLABLE POSITION INTERACTIONS",
                df_to_use=main_df
            )

            print("\n--- Model 3d execution complete. ---")

        except Exception as e:
            print(f"\n!!! An unexpected error occurred during Model 3d execution: {e} !!!")
            import traceback; traceback.print_exc()
    else:
        print("!!! ERROR: 'main_df' not found. Please run the data loading cells first. !!!")

In [ ]:
# ==============================================================================
# 30. EXECUTE ITERATIVE MODEL BUILDING: STEP 17 (COMPARE MODEL 3c vs MODEL 3d)
# ==============================================================================

if __name__ == '__main__':
    # We are comparing our previous mixed model (Model 3c) against this final one.
    model_to_compare_1 = "Model_3c_CueTrading_Interactions"
    model_to_compare_2 = "Model_3d_Syllable_Position_Interactions"

    compare_brms_models_subprocess(model_to_compare_1, model_to_compare_2)

In [ ]:
"""¡¡¡¡¡DEPRECATED. DO NOT RUN THIS MODEL!!!!!"""
"""¡¡¡¡¡DEPRECATED. DO NOT RUN THIS MODEL!!!!!"""
"""¡¡¡¡¡DEPRECATED. DO NOT RUN THIS MODEL!!!!!"""
# ==============================================================================
# 33. EXECUTE ITERATIVE MODEL BUILDING: STEP 18 (MODEL 4 - Final (All Interactions))
# ==============================================================================

if __name__ == '__main__':
    if 'main_df' in locals() and main_df is not None:
        try:
            # --- 1. Define the formula for Model 4 ---
            # This formula builds directly on the structure of Model 2.

            # Base fixed effects
            fixed_effects_formula = """
                Duration_z + Pitch_z + Intensity_Q2_z + Spectral_Tilt_DB_z +
                F1_z + F2_z + Pace_SPS_z + Syllable_position_z + Syllable_Index_z +
                Phrase_Position + Is_Interrogative + Is_Exclamative +
                Preceded_by_Silence + Followed_by_Silence + Vowel
            """

            # Base random effects from Model 2
            random_effects_intercepts = "(1|Words)"
            random_slopes_for_speaker = "(1 + Duration_z + Pitch_z + Spectral_Tilt_DB_z | Speaker)"

            # Define Vowel Quality Interactions
            vowel_quality = """
                F1_z * Vowel
            """

            # Define the Cue Trading and Vowel Quality Interactions
            cue_trading_interactions = """
                Duration_z * Pitch_z * Spectral_Tilt_DB_z
            """

            # Define the Positional Interaction Terms
            positional_interactions = """
                Duration_z * Syllable_position_z +
                Spectral_Tilt_DB_z * Syllable_position_z +
                Duration_z * Syllable_Index_z +
                Spectral_Tilt_DB_z * Syllable_Index_z
            """

            # Combine all parts into the final formula for brms
            model_4_formula = (f"bf(Matteson_Stress ~ {fixed_effects_formula} + "
                              f"{vowel_quality} + "
                              f"{cue_trading_interactions} + "
                              f"{positional_interactions} + "
                              f"{random_effects_intercepts} + {random_slopes_for_speaker})")

            # --- 2. Run and Save Model 4 ---
            run_brms_subprocess(
                model_formula=model_4_formula,
                model_name="Model_4_All_Interactions",
                output_file=BRM_OUTPUT_FILE,
                model_description="MODEL 4: ALL INTERACTIONS (ADDING POSITIONAL INTERACTIONS)",
                df_to_use=main_df
            )

            print("\n--- Model 4 execution complete. ---")

        except Exception as e:
            print(f"\n!!! An unexpected error occurred during Model 4 execution: {e} !!!")
            import traceback; traceback.print_exc()
    else:
        print("!!! ERROR: 'main_df' not found. Please run the data loading cells first. !!!")

In [ ]:
# ==============================================================================
# 34. EXECUTE ITERATIVE MODEL BUILDING: STEP 19 (COMPARE MODEL 3d vs MODEL 4)
# ==============================================================================

if __name__ == '__main__':
    # We are comparing our previous mixed model (Model 3c) against this final one.
    model_to_compare_1 = "Model_3d_Syllable_Position_Interactions"
    model_to_compare_2 = "Model_4_All_Interactions"

    compare_brms_models_subprocess(model_to_compare_1, model_to_compare_2)

In [ ]:
# ==============================================================================
# 37. BINARY MODEL A: PRIMARY vs. SECONDARY STRESS
# ==============================================================================
import pandas as pd

if __name__ == '__main__':
    if 'main_df' in locals() and main_df is not None:
        try:
            print("\n--- Preparing Data for Binary Model A (Primary vs. Secondary) ---")

            # 1. Filter Data: Keep only Primary and Secondary
            target_classes = ['Primary_stress', 'Secondary_stress']
            df_binary_a = main_df[main_df['Matteson_Stress'].isin(target_classes)].copy()

            # 2. Map to Binary (0/1)
            # Target: Distinguish Primary (1) from Secondary (0)
            df_binary_a['Binary_Outcome'] = df_binary_a['Matteson_Stress'].apply(
                lambda x: 1 if x == 'Primary_stress' else 0
            )

            print(f"  - Subset created: {len(df_binary_a)} rows.")
            
            # 3. Define Formula (Full Interaction Structure)
            fixed_effects_formula = """
                Duration_z + Pitch_z + Intensity_Q2_z + Spectral_Tilt_DB_z +
                F1_z + F2_z + Pace_SPS_z + Syllable_position_z + Syllable_Index_z +
                Phrase_Position + Is_Interrogative + Is_Exclamative +
                Preceded_by_Silence + Followed_by_Silence + Vowel
            """
            random_effects = "(1|Words) + (1 + Duration_z + Pitch_z + Spectral_Tilt_DB_z | Speaker)"
            
            # Interactions
            vowel_quality = "F1_z * Vowel"
            cue_trading_interactions = "Duration_z * Pitch_z * Spectral_Tilt_DB_z"
            positional_interactions = """
                Duration_z * Syllable_position_z +
                Spectral_Tilt_DB_z * Syllable_position_z
            """

            # Note: Dependent variable is 'Binary_Outcome'
            model_binary_formula_a = f"bf(Binary_Outcome ~ {fixed_effects_formula} + {vowel_quality} + {cue_trading_interactions}  + {positional_interactions}+ {random_effects})"

            # 4. Run Model
            run_brms_subprocess(
                model_formula=model_binary_formula_a,
                model_name="Model_Binary_A_Primary_vs_Secondary",
                output_file=BRM_OUTPUT_FILE,
                model_description="BINARY MODEL A: PRIMARY (1) vs. SECONDARY (0)",
                df_to_use=df_binary_a,       # <--- Pass the subset!
                family="bernoulli('logit')"  # <--- CRITICAL override
            )

        except Exception as e:
            print(f"\n!!! An unexpected error occurred: {e} !!!")
            import traceback; traceback.print_exc()
    else:
        print("!!! ERROR: Prerequisite steps not complete. !!!")

In [ ]:
# ==============================================================================
# 38. BINARY MODEL B: SECONDARY vs. UNSTRESSED
# ==============================================================================
import pandas as pd

if __name__ == '__main__':
    if 'main_df' in locals() and main_df is not None:
        try:
            print("\n--- Preparing Data for Binary Model B (Secondary vs. Unstressed) ---")

            # 1. Filter Data: Keep only Secondary and Unstressed
            target_classes = ['Secondary_stress', 'Unstressed']
            df_binary_b = main_df[main_df['Matteson_Stress'].isin(target_classes)].copy()

            # 2. Map to Binary (0/1)
            # Target: Distinguish Secondary (1) from Unstressed (0)
            df_binary_b['Binary_Outcome'] = df_binary_b['Matteson_Stress'].apply(
                lambda x: 1 if x == 'Secondary_stress' else 0
            )

            print(f"  - Subset created: {len(df_binary_b)} rows.")

            # 3. Define Formula (Full Interaction Structure)
            fixed_effects_formula = """
                Duration_z + Pitch_z + Intensity_Q2_z + Spectral_Tilt_DB_z +
                F1_z + F2_z + Pace_SPS_z + Syllable_position_z + Syllable_Index_z +
                Phrase_Position + Is_Interrogative + Is_Exclamative +
                Preceded_by_Silence + Followed_by_Silence + Vowel
            """
            random_effects = "(1|Words) + (1 + Duration_z + Pitch_z + Spectral_Tilt_DB_z | Speaker)"
            
            vowel_quality = "F1_z * Vowel"
            cue_trading_interactions = "Duration_z * Pitch_z * Spectral_Tilt_DB_z"
            positional_interactions = """
                Duration_z * Syllable_position_z +
                Spectral_Tilt_DB_z * Syllable_position_z
            """

            model_binary_formula_b = f"bf(Binary_Outcome ~ {fixed_effects_formula} + {vowel_quality} + {cue_trading_interactions} + {positional_interactions} + {random_effects})"

            # 4. Run Model
            run_brms_subprocess(
                model_formula=model_binary_formula_b,
                model_name="Model_Binary_B_Secondary_vs_Unstressed",
                output_file=BRM_OUTPUT_FILE,
                model_description="BINARY MODEL B: SECONDARY (1) vs. UNSTRESSED (0)",
                df_to_use=df_binary_b,       # <--- Pass the subset!
                family="bernoulli('logit')"  # <--- CRITICAL override
            )

        except Exception as e:
            print(f"\n!!! An unexpected error occurred: {e} !!!")
            import traceback; traceback.print_exc()
    else:
        print("!!! ERROR: Prerequisite steps not complete. !!!")

In [ ]:
# ==============================================================================
# 39. VISUALIZE SIGNIFICANT INTERACTIONS
# ==============================================================================

# INSTRUCTIONS:
# For each significant interaction you identified in the model comparison steps,
# uncomment and run a block of code like the one below.

if __name__ == '__main__':
    # --- Example: Visualize the Duration * Syllable Position Interaction ---
    
    model_name_to_plot = "Model_4_All_Interactions"
    interaction_to_plot = "Duration_z:Syllable_position_z"
    output_filename = "Interaction_Plot_Duration_by_Position.png"

    plot_conditional_effects_subprocess(
        model_name=model_name_to_plot,
        effect_vars=interaction_to_plot,
        plot_filename=output_filename
    )

    print("\n--- Visualization section ready. Uncomment lines above to run. ---")

In [ ]:
# ==============================================================================
# 40. MODEL CRITICISM: POSTERIOR PREDICTIVE CHECKS
# ==============================================================================

# INSTRUCTIONS:
# Run this block after you have identified your "best" model.

if __name__ == '__main__':
    # --- Example: Check Model 4 ---
    
    model_to_check = "Model_Binary_B_Secondary_vs_Unstressed"
    output_filename = "PPC_BarPlot_Binary_B_SvU.png"

    run_ppc_subprocess(
        model_name=model_to_check,
        plot_filename=output_filename
    )

    print("\n--- Model Criticism section ready. Uncomment lines above to run. ---")

## Execution Blocks for Random Forest Models

In [ ]:
# ==============================================================================
# 40a. RANDOM FOREST DATA PREPARATION: STRATIFIED SAMPLING
# ==============================================================================
# Rationale: Running conditional variable importance on 190k rows with high 
# correlation requires >100GB RAM. We use a representative stratified sample 
# to stabilize rankings within hardware limits (24GB RAM).
# ==============================================================================
import pandas as pd

if 'main_df' in locals() and main_df is not None:
    # --- CONFIGURATION ---
    # Set to 10000 for Hyperparameter Tuning (ntree/mtry tests)
    # Set to 50000 for Final Publication Run
    RF_SAMPLE_SIZE = 20000 
    
    print(f"--- Generating Stratified Sample (N={RF_SAMPLE_SIZE}) ---")
    
    # 1. Perform Stratified Sampling
    # Group by Stress to preserve the 13% Secondary Stress ratio
    df_rf_sample = main_df.groupby('Matteson_Stress', group_keys=False).apply(
        lambda x: x.sample(min(len(x), int(RF_SAMPLE_SIZE * (len(x)/len(main_df)))), random_state=42)
    )
    
    # 2. Verification
    print(f"Original Rows: {len(main_df)}")
    print(f"Sampled Rows:  {len(df_rf_sample)}")
    
    print("\n--- Class Distribution Check (Verification) ---")
    orig_dist = main_df['Matteson_Stress'].value_counts(normalize=True)
    samp_dist = df_rf_sample['Matteson_Stress'].value_counts(normalize=True)
    
    compare_df = pd.DataFrame({'Original %': orig_dist, 'Sample %': samp_dist})
    print(compare_df)
    
    print("\n> 'df_rf_sample' is ready for use in RF models.")
else:
    print("!!! Error: main_df not found. Run data loading first. !!!")

In [ ]:
# ==============================================================================
# 41. EXECUTE RF: FULL MODEL - GLOBAL (ALL STRESS LEVELS)
# ==============================================================================
if __name__ == '__main__':
    if 'df_rf_sample' in locals():
        try:
            full_predictors = [
                'Duration_z', 'Pitch_z', 'Intensity_Q2_z', 'Spectral_Tilt_DB_z',
                'F1_z', 'F2_z', 'Pace_SPS_z',
                'Syllable_position_z', 'Syllable_Index_z',
                'Phrase_Position', 'Is_Interrogative', 'Is_Exclamative',
                'Preceded_by_Silence', 'Followed_by_Silence', 'Vowel'
            ]

            run_cforest_subprocess(
                output_file=RF_FULL_OUTPUT_FILE,
                model_label="Full_Global",
                predictors_list=full_predictors,
                df_to_use=df_rf_sample, # <--- Uses the Stratified Sample
                mtry_val=12,             # Starting with sqrt(15) approx 4
                ntree_val=1000
            )
        except Exception as e:
            print(f"\n!!! Error: {e} !!!"); import traceback; traceback.print_exc()
    else:
        print("!!! ERROR: 'df_rf_sample' not found. Please run Cell 40a first. !!!")

In [ ]:
# ==============================================================================
# 42. EXECUTE RF: FULL MODEL - BINARY A (PRIMARY vs SECONDARY)
# ==============================================================================
if __name__ == '__main__':
    if 'df_rf_sample' in locals():
        try:
            print("\n--- Preparing Data for RF Binary A ---")
            target_classes = ['Primary_stress', 'Secondary_stress']
            df_bin_a = df_rf_sample[df_rf_sample['Matteson_Stress'].isin(target_classes)].copy()
            
            full_predictors = [
                'Duration_z', 'Pitch_z', 'Intensity_Q2_z', 'Spectral_Tilt_DB_z',
                'F1_z', 'F2_z', 'Pace_SPS_z',
                'Syllable_position_z', 'Syllable_Index_z',
                'Phrase_Position', 'Is_Interrogative', 'Is_Exclamative',
                'Preceded_by_Silence', 'Followed_by_Silence', 'Vowel'
            ]

            run_cforest_subprocess(
                output_file=RF_FULL_OUTPUT_FILE,
                model_label="Full_Binary_A_Pri_vs_Sec",
                predictors_list=full_predictors,
                df_to_use=df_bin_a,
                mtry_val=12,
                ntree_val=1000
            )
        except Exception as e:
            print(f"\n!!! Error: {e} !!!"); import traceback; traceback.print_exc()
    else:
        print("!!! ERROR: 'df_rf_sample' not found. !!!")

In [ ]:
# ==============================================================================
# 43. EXECUTE RF: FULL MODEL - BINARY B (SECONDARY vs UNSTRESSED)
# ==============================================================================
if __name__ == '__main__':
    if 'df_rf_sample' in locals():
        try:
            print("\n--- Preparing Data for RF Binary B ---")
            target_classes = ['Secondary_stress', 'Unstressed']
            df_bin_b = df_rf_sample[df_rf_sample['Matteson_Stress'].isin(target_classes)].copy()

            full_predictors = [
                'Duration_z', 'Pitch_z', 'Intensity_Q2_z', 'Spectral_Tilt_DB_z',
                'F1_z', 'F2_z', 'Pace_SPS_z',
                'Syllable_position_z', 'Syllable_Index_z',
                'Phrase_Position', 'Is_Interrogative', 'Is_Exclamative',
                'Preceded_by_Silence', 'Followed_by_Silence', 'Vowel'
            ]

            run_cforest_subprocess(
                output_file=RF_FULL_OUTPUT_FILE,
                model_label="Full_Binary_B_Sec_vs_Uns",
                predictors_list=full_predictors,
                df_to_use=df_bin_b,
                mtry_val=12,
                ntree_val=1000
            )
        except Exception as e:
            print(f"\n!!! Error: {e} !!!"); import traceback; traceback.print_exc()
    else:
        print("!!! ERROR: 'df_rf_sample' not found. !!!")

In [ ]:
# ==============================================================================
# 44. EXECUTE RF: ACOUSTIC ONLY - GLOBAL (ALL STRESS LEVELS)
# ==============================================================================
if __name__ == '__main__':
    if 'df_rf_sample' in locals():
        try:
            acoustic_predictors = [
                'Duration_z', 'Pitch_z', 'Intensity_Q2_z', 'Spectral_Tilt_DB_z',
                'F1_z', 'F2_z', 'Vowel'
            ]

            run_cforest_subprocess(
                output_file=RF_ACOUSTIC_OUTPUT_FILE,
                model_label="Acoustic_Global",
                predictors_list=acoustic_predictors,
                df_to_use=df_rf_sample,
                mtry_val=5,
                ntree_val=1000
            )
        except Exception as e:
            print(f"\n!!! Error: {e} !!!"); import traceback; traceback.print_exc()
    else:
        print("!!! ERROR: 'df_rf_sample' not found. !!!")

In [ ]:
# ==============================================================================
# 45. EXECUTE RF: ACOUSTIC ONLY - BINARY A (PRIMARY vs SECONDARY)
# ==============================================================================
if __name__ == '__main__':
    if 'df_rf_sample' in locals():
        try:
            print("\n--- Preparing Data for RF Acoustic Binary A ---")
            target_classes = ['Primary_stress', 'Secondary_stress']
            df_bin_a = df_rf_sample[df_rf_sample['Matteson_Stress'].isin(target_classes)].copy()

            acoustic_predictors = [
                'Duration_z', 'Pitch_z', 'Intensity_Q2_z', 'Spectral_Tilt_DB_z',
                'F1_z', 'F2_z', 'Vowel'
            ]

            run_cforest_subprocess(
                output_file=RF_ACOUSTIC_OUTPUT_FILE,
                model_label="Acoustic_Binary_A_Pri_vs_Sec",
                predictors_list=acoustic_predictors,
                df_to_use=df_bin_a,
                mtry_val=5,
                ntree_val=1000
            )
        except Exception as e:
            print(f"\n!!! Error: {e} !!!"); import traceback; traceback.print_exc()
    else:
        print("!!! ERROR: 'df_rf_sample' not found. !!!")

In [ ]:
# ==============================================================================
# 46. EXECUTE RF: ACOUSTIC ONLY - BINARY B (SECONDARY vs UNSTRESSED)
# ==============================================================================
if __name__ == '__main__':
    if 'df_rf_sample' in locals():
        try:
            print("\n--- Preparing Data for RF Acoustic Binary B ---")
            target_classes = ['Secondary_stress', 'Unstressed']
            df_bin_b = df_rf_sample[df_rf_sample['Matteson_Stress'].isin(target_classes)].copy()

            acoustic_predictors = [
                'Duration_z', 'Pitch_z', 'Intensity_Q2_z', 'Spectral_Tilt_DB_z',
                'F1_z', 'F2_z', 'Vowel'
            ]

            run_cforest_subprocess(
                output_file=RF_ACOUSTIC_OUTPUT_FILE,
                model_label="Acoustic_Binary_B_Sec_vs_Uns",
                predictors_list=acoustic_predictors,
                df_to_use=df_bin_b,
                mtry_val=5,
                ntree_val=1000
            )
        except Exception as e:
            print(f"\n!!! Error: {e} !!!"); import traceback; traceback.print_exc()
    else:
        print("!!! ERROR: 'df_rf_sample' not found. !!!")

In [ ]:
print("--- CPU CORES AVAILABLE ---")
!nproc

print("\n--- MEMORY (RAM) LIMITS ---")
!free -h

In [ ]:
# ==============================================================================
# DIAGNOSTIC TOOL: RF COMPLEXITY CHECKER (FIXED)
# ==============================================================================
import subprocess
import os
import pandas as pd

def run_rf_diagnostic(model_label, predictors_list, df_to_use, mtry_val=5):
    print(f"\n----- DIAGNOSTIC RUN: {model_label} -----")
    
    # Paths
    temp_data = os.path.join(MODEL_OUTPUT_FOLDER, f"temp_diag_{model_label}.csv")
    temp_script = os.path.join(MODEL_OUTPUT_FOLDER, f"temp_diag_{model_label}.R")
    
    # Save Data
    df_to_use.to_csv(temp_data, index=False)
    
    pred_vec = "c('" + "', '".join(predictors_list) + "')"
    
    r_content = f"""
    suppressPackageStartupMessages(library(partykit))
    
    # 1. Load Data
    df <- read.csv("{temp_data}")
    
    # --- DATA PREPARATION (The Missing Part) ---
    
    # Target Handling
    df$Matteson_Stress <- factor(df$Matteson_Stress)
    possible_levels <- c('Unstressed', 'Secondary_stress', 'Primary_stress')
    actual_levels <- intersect(possible_levels, levels(df$Matteson_Stress))
    if(length(actual_levels) > 0) {{
         df$Matteson_Stress <- factor(df$Matteson_Stress, levels = actual_levels, ordered = TRUE)
    }}
    
    # Predictor Factor Handling (Fixes 'character' error for Vowel)
    cat_vars <- c('Phrase_Position', 'Is_Interrogative', 'Is_Exclamative',
                  'Preceded_by_Silence', 'Followed_by_Silence', 'Vowel')
    for (var in cat_vars) {{
        if (var %in% names(df)) {{ df[[var]] <- as.factor(df[[var]]) }}
    }}
    
    cat("  > Data Loaded. Rows:", nrow(df), "\\n")
    
    # 2. Fit a Mini-Forest (50 Trees)
    cat("  > Fitting Mini-Forest (50 trees) to check structure...\\n")
    set.seed(42)
    ctrl <- ctree_control(teststat = "quad", testtype = "Univ", mincriterion = 0, saveinfo = FALSE)
    
    rf <- cforest(Matteson_Stress ~ ., 
                  data = df[,c("Matteson_Stress", {pred_vec})], 
                  ntree = 50, 
                  mtry = {mtry_val},
                  perturb = list(replace = FALSE, fraction = 0.632),
                  control = ctrl,
                  cores = 10)
    
    # 3. Analyze Tree Complexity
    cat("\\n--- TREE STRUCTURE EVIDENCE ---\\n")
    
    depths <- integer(50)
    widths <- integer(50)
    
    for(i in 1:50) {{
        tr <- gettree(rf, i)
        depths[i] <- max(depth(tr))
        widths[i] <- width(tr)
    }}
    
    cat("  Average Tree Depth:", mean(depths), "\\n")
    cat("  Max Tree Depth:    ", max(depths), "\\n")
    cat("  Average Leaves:    ", mean(widths), "\\n")
    cat("  (Note: Deep trees + High Correlation = Exponentially slower conditional importance)\\n")
    
    # 4. Analyze Correlation
    cat("\\n--- CORRELATION EVIDENCE ---\\n")
    
    # Subset predictors
    preds <- df[, {pred_vec}]
    
    # Convert factors to numeric for correlation check
    preds_num <- data.frame(lapply(preds, function(x) as.numeric(x)))
    
    # Calculate Correlation Matrix (Spearman for mixed types)
    cor_mat <- cor(preds_num, method = "spearman")
    print(round(cor_mat, 2))
    """
    
    with open(temp_script, "w") as f: f.write(r_content)
    
    try:
        result = subprocess.run(["Rscript", temp_script], check=True, capture_output=True, text=True)
        print(result.stdout)
    except subprocess.CalledProcessError as e:
        print("!!! Diagnostic Failed !!!")
        print(e.stderr)

In [ ]:
if __name__ == '__main__':
    if 'df_rf_sample' in locals():
        # The predictors for the problematic model
        acoustic_predictors = [
            'Duration_z', 'Pitch_z', 'Intensity_Q2_z', 'Spectral_Tilt_DB_z',
            'F1_z', 'F2_z', 'Vowel'
        ]

        run_rf_diagnostic(
            model_label="DIAGNOSTIC_Acoustic_Global",
            predictors_list=acoustic_predictors,
            df_to_use=df_rf_sample,
            mtry_val=4 # Using the mtry value you used when it stalled
        )

In [ ]:
if __name__ == '__main__':
    if 'df_rf_sample' in locals():
        # The 15 predictors used in the Full Global Model
        full_predictors = [
            'Duration_z', 'Pitch_z', 'Intensity_Q2_z', 'Spectral_Tilt_DB_z',
            'F1_z', 'F2_z', 'Pace_SPS_z',
            'Syllable_position_z', 'Syllable_Index_z',
            'Phrase_Position', 'Is_Interrogative', 'Is_Exclamative',
            'Preceded_by_Silence', 'Followed_by_Silence', 'Vowel'
        ]

        run_rf_diagnostic(
            model_label="DIAGNOSTIC_Full_Global",
            predictors_list=full_predictors,
            df_to_use=df_rf_sample,
            mtry_val=12 # Using the high mtry you used for the Full Model
        )